# Scraping Toyosu fish market HTML documents

Toyosu Weekly fish martket report has prices for Bluefin tuna ("tuna") and Bigeye tuna ("bigeye") from the years 2004 -> 2023 in scrape-able html files. Tables should be consistently formatted up to 2023. After 2023 html file becomes pdf document, and I could manually collect the last two or so years.

Data is weekly.

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import re
import pandas as pd

In [ ]:
#Scrape - trying the first page just to check if it works
    #first scrapable page is January 2023 Week 1

url = "https://www.spt.metro.tokyo.lg.jp/shijou/torihiki/03/suisan/2023013.html"
raw_html = requests.get(url).content
soup_doc = BeautifulSoup(raw_html, "html.parser")
print(soup_doc.prettify())

#content is scrapable!


Making a scraping script that will scrape links with the following format: https://www.spt.metro.tokyo.lg.jp/shijou/torihiki/03/suisan/2004011.html
each website is has a table with information I would like to scrape. 

I don't want to get IP blocked so please add a delay for the process with python's time library, with delay=.25

Each weekly report has the same url with a code denoting date at the back. Format is YYYYMMW (Year, Month, Week) Example: 2004011 - 2004, 01 (January), 1 (week 1) 4 weeks is standard, but some have 5 weeks. 
Try 4 weeks, try the fifth and if that gets a "Not Found," don't add data to the table and proceed to the next month. 

Years: 2004 to 2023
Months: 01 to 12
Weeks: 1 to 4 or 5
I want to pull the following rows: 

マグロ (row named マグロ, and the following row)
メバチ (row named メバチ, and only that one) which can be accomplished with regex, probably

These rows are found within /html/body/center/table/tbody/tr/td/table[3] (the third table element within the outermost table element)
data pulled from each row should be put into the final columns, where data from each cell is divided with a pipe |

save each row to a pd dataframe

In [37]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

BASE_URL = "https://www.spt.metro.tokyo.lg.jp/shijou/torihiki/03/suisan/{}.html"

# Delay between requests (seconds)
REQUEST_DELAY = 0.1

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# ── Column definitions ────────────────────────────────────────────────────────
#
# The detail table (section 2) has these columns left-to-right:
#
#  [ITEM BLOCK]          [VOLUME BLOCK]                    [PRICE BLOCK]
#  品目 | 取扱数量t | 週↑↓ | 前週比 | 前年同期比 | 産地 | 等級 | 高値 | 中値 | 安値 | 前週比 | 前年同期比 | (サイズ)
#   0        1         2       3          4          5      6      7      8      9      10         11          12
#
# First sub-row of an item has all 13 cells.
# Continuation sub-rows (extra origin/grade/price combos) only have 8 cells:
#   産地 | 等級 | 高値 | 中値 | 安値 | 前週比 | 前年同期比 | (サイズ)
#    0      1      2      3      4       5          6           7

FULL_ROW_COLS = [
    "品目",                # 0  item name
    "取扱数量t",           # 1  volume (tonnes)
    "週の状況",            # 2  week trend symbol (↑/↓/→ etc.)
    "increase_decrease_1", # 3  placeholder for alignment
    "前週比",              # 4  vs previous week (%)
    "increase_decrease_2", # 5  placeholder for alignment
    "前年同期比",          # 6  vs same period last year (%)
    "産地",                # 7  origin / fishing ground
    "等級",                # 8  grade / size class
    "高値",                # 9  price high (¥/kg)
    "中値",                # 10 price mid  (¥/kg)
    "安値",                # 11 price low  (¥/kg)
    "前週比_価格",         # 12 price vs prev week (%)
    "前年同期比_価格",     # 13 price vs prev year (%)
    "サイズ",              # 14 size note
]

SUB_ROW_COLS = [        # continuation rows: price block only
    "産地",
    "等級",
    "高値",
    "中値",
    "安値",
    "前週比_価格",
    "前年同期比_価格",
    "サイズ",
]

META_COLS = ["year", "month", "week", "code", "url", "row_type", "sub_row"]
ALL_COLS  = META_COLS + FULL_ROW_COLS


# ── Helpers ───────────────────────────────────────────────────────────────────

def build_url_code(year, month, week):
    return f"{year}{month:02d}{week}"


def fetch_page(url):
    """Fetch page decoded as Shift-JIS. Returns HTML string or None if not found."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code == 404:
            return None
        resp.raise_for_status()
        resp.encoding = "shift_jis"
        return resp.text
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            return None
        print(f"  HTTP error for {url}: {e}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  Request error for {url}: {e}")
        return None


def blank_full_row():
    return {col: "" for col in FULL_ROW_COLS}


def map_full_row(cells):
    """Map a 13-cell row to FULL_ROW_COLS dict."""
    row = blank_full_row()
    for i, col in enumerate(FULL_ROW_COLS):
        if i < len(cells):
            row[col] = cells[i]
    return row


def map_sub_row(cells, parent_item):
    """Map an 8-cell continuation row; inherit item name from parent."""
    row = blank_full_row()
    row["品目"] = parent_item
    for i, col in enumerate(SUB_ROW_COLS):
        if i < len(cells):
            row[col] = cells[i]
    return row


# ── Parser ────────────────────────────────────────────────────────────────────

def parse_table(html):
    """
    Return the last 3 rows from the table: マグロ, マグロ_next, and メバチ.
    Each dict has keys from FULL_ROW_COLS plus 'row_type' and 'sub_row'.

    row_type values:
      'マグロ'       – first (main) row of マグロ block
      'マグロ_next'  – the ONE sub-row immediately following マグロ
      'メバチ'       – メバチ row
    """
    soup = BeautifulSoup(html, "html.parser")
    all_trs = soup.find_all("tr")
    records = []

    i = 0
    while i < len(all_trs):
        tds   = all_trs[i].find_all(["td", "th"])
        cells = [td.get_text(strip=True) for td in tds]

        is_maguro  = any(re.fullmatch(r"マグロ", c) for c in cells)
        is_mebachi = any(re.fullmatch(r"メバチ", c) for c in cells)

        # ── マグロ block ──────────────────────────────────────────────────────
        if is_maguro:
            # Main row
            records.append({
                "row_type": "マグロ",
                "sub_row":  0,
                **map_full_row(cells),
            })
            i += 1

            # Collect ONLY the ONE immediately following sub-row
            if i < len(all_trs):
                ntds  = all_trs[i].find_all(["td", "th"])
                nc    = [td.get_text(strip=True) for td in ntds]
                nn    = len(nc)

                # If next row exists and is not empty, capture it as マグロ_next
                if nn > 0:
                    first = nc[0]
                    # Only capture if it's a continuation (not a new named item)
                    if not (nn >= 13 and first and first not in ("", "マグロ")):
                        records.append({
                            "row_type": "マグロ_next",
                            "sub_row":  1,
                            **map_sub_row(nc, "マグロ"),
                        })
                        i += 1

            continue  # don't increment i again

        # ── メバチ (single row only) ──────────────────────────────────────────
        if is_mebachi:
            records.append({
                "row_type": "メバチ",
                "sub_row":  0,
                **map_full_row(cells),
            })

        i += 1

    # Return only the last 3 rows (skip the first ones if multiple sets exist)
    return records[-3:] if len(records) >= 3 else records


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    all_records = []

    years     = range(2004, 2005)   # 2004–2023 inclusive
    months    = range(1, 4)
    max_weeks = 5

    for year in years:
        for month in months:
            for week in range(1, max_weeks + 1):
                code = build_url_code(year, month, week)
                url  = BASE_URL.format(code)

                print(f"Fetching {code} ...", end=" ", flush=True)
                html = fetch_page(url)

                if html is None:
                    msg = "Not found (week 5 — skipping)" if week == 5 else "Not found"
                    print(msg)
                    time.sleep(REQUEST_DELAY)
                    continue

                if re.search(r"not found|404|ページが見つかり", html, re.IGNORECASE):
                    print("Soft 404 (skipping)")
                    time.sleep(REQUEST_DELAY)
                    continue

                rows = parse_table(html)

                if not rows:
                    print("No matching rows")
                else:
                    print(f"{len(rows)} row(s) captured")
                    for row in rows:
                        record = {
                            "year":     year,
                            "month":    month,
                            "week":     week,
                            "code":     code,
                            "url":      url,
                            "row_type": row.pop("row_type"),
                            "sub_row":  row.pop("sub_row"),
                        }
                        record.update(row)
                        all_records.append(record)

                time.sleep(REQUEST_DELAY)

    df = pd.DataFrame(all_records, columns=ALL_COLS)
    print(f"\nDone! {len(df)} rows collected.")
    print(df.head())
    return df


if __name__ == "__main__":
    df = main()

Fetching 2004011 ... 3 row(s) captured
Fetching 2004012 ... 3 row(s) captured
Fetching 2004013 ... 3 row(s) captured
Fetching 2004014 ... 3 row(s) captured
Fetching 2004015 ... Not found (week 5 — skipping)
Fetching 2004021 ... 3 row(s) captured
Fetching 2004022 ... 3 row(s) captured
Fetching 2004023 ... 3 row(s) captured
Fetching 2004024 ... 3 row(s) captured
Fetching 2004025 ... Not found (week 5 — skipping)
Fetching 2004031 ... 3 row(s) captured
Fetching 2004032 ... 3 row(s) captured
Fetching 2004033 ... 3 row(s) captured
Fetching 2004034 ... 3 row(s) captured
Fetching 2004035 ... Not found (week 5 — skipping)

Done! 36 rows collected.
   year  month  week     code  \
0  2004      1     1  2004011   
1  2004      1     1  2004011   
2  2004      1     1  2004011   
3  2004      1     2  2004012   
4  2004      1     2  2004012   

                                                 url  row_type  sub_row   品目  \
0  https://www.spt.metro.tokyo.lg.jp/shijou/torih...       マグロ        0  マ

In [38]:
df.head(3)

,year,month,week,code,url,row_type,sub_row,品目,取扱数量t,週の状況,...,increase_decrease_2,前年同期比,産地,等級,高値,中値,安値,前週比_価格,前年同期比_価格,サイズ
0,2004,1,1,2004011,https://www.spt.metro.tokyo.lg.jp/shijou/torih...,マグロ,0,マグロ,49.2,74,...,↑,各地,生,"27,300","6,009","2,100",108,94,,
1,2004,1,1,2004011,https://www.spt.metro.tokyo.lg.jp/shijou/torih...,マグロ_next,1,マグロ,,,...,,,海外,生,"9,450","2,987","1,890",112,76,
2,2004,1,1,2004011,https://www.spt.metro.tokyo.lg.jp/shijou/torih...,メバチ,0,メバチ,97.9,60,...,↓,各地*,冷凍,"4,200","1,095",578,100,111,,


In [39]:
# ── Post-processing ───────────────────────────────────────────────────────────

def post_process_dataframe(df):
    """
    Post-process the dataframe to:
    1. Translate Japanese labels to English
    2. Fix alignment of マグロ_next rows (shift data one position right)
    
    Returns a new dataframe with these adjustments applied.
    """
    # Translation mapping
    ja_to_en = {
        "品目": "item",
        "取扱数量t": "volume_tonnes",
        "週の状況": "week_trend",
        "increase_decrease_1": "increase_decrease_1",
        "前週比": "vs_prev_week_pct",
        "increase_decrease_2": "increase_decrease_2",
        "前年同期比": "vs_prev_year_pct",
        "産地": "origin",
        "等級": "grade",
        "高値": "price_high",
        "中値": "price_mid",
        "安値": "price_low",
        "前週比_価格": "price_vs_prev_week_pct",
        "前年同期比_価格": "price_vs_prev_year_pct",
        "サイズ": "size",
    }
    
    row_type_map = {
        "マグロ": "tuna",
        "マグロ_next": "tuna_next",
        "メバチ": "bigeye",
    }
    
    df_copy = df.copy()
    
    # Rename columns from Japanese to English
    rename_cols = {col: ja_to_en.get(col, col) for col in df_copy.columns}
    df_copy = df_copy.rename(columns=rename_cols)
    
    # Translate row_type values
    df_copy["row_type"] = df_copy["row_type"].map(row_type_map)
    
    # Fix alignment for tuna and bigeye rows: shift data one position right starting at "vs_prev_year_pct"
    data_cols = [col for col in df_copy.columns if col not in META_COLS]
    
    # Find the index where "vs_prev_year_pct" starts
    shift_start_idx = data_cols.index("vs_prev_year_pct") if "vs_prev_year_pct" in data_cols else None
    
    if shift_start_idx is not None:
        cols_to_shift = data_cols[shift_start_idx:]
        
        for idx in df_copy.index:
            if df_copy.loc[idx, "row_type"] in ["tuna", "bigeye"]:
                # Get the values for columns that need shifting
                values = df_copy.loc[idx, cols_to_shift].tolist()
                
                # Shift right by one: prepend empty string and remove last
                shifted_values = [""] + values[:-1]
                
                # Put shifted values back
                df_copy.loc[idx, cols_to_shift] = shifted_values
    
    return df_copy


In [40]:
df = post_process_dataframe(df)
#drop url and size columns

In [41]:
#drop url, size, and vs_prev_year_pct columns
df = df.drop(columns=["url", "size", "vs_prev_year_pct"])
#rename vs_prev_week_pct to year-over-year-comparison
df = df.rename(columns={"vs_prev_week_pct": "year_over_year_comparison"})


In [43]:
df.head(10)

,year,month,week,code,row_type,sub_row,item,volume_tonnes,week_trend,increase_decrease_1,year_over_year_comparison,increase_decrease_2,origin,grade,price_high,price_mid,price_low,price_vs_prev_week_pct,price_vs_prev_year_pct
0,2004,1,1,2004011,tuna,0,マグロ,49.2,74,↓,112,↑,各地,生,"27,300","6,009","2,100",108,94
1,2004,1,1,2004011,tuna_next,1,マグロ,,,,,,海外,生,"9,450","2,987","1,890",112,76
2,2004,1,1,2004011,bigeye,0,メバチ,97.9,60,↓,82,↓,各地*,冷凍,"4,200","1,095",578,100,111
3,2004,1,2,2004012,tuna,0,マグロ,35.8,73,↓,100,,各地,生,"8,400","5,384","2,625",90,66
4,2004,1,2,2004012,tuna_next,1,マグロ,,,,,,海外,生,"5,775","2,387","1,785",80,66
5,2004,1,2,2004012,bigeye,0,メバチ,97.9,100,,85,↓,各地*,冷凍,"4,200","1,055",630,96,111
6,2004,1,3,2004013,tuna,0,マグロ,30.0,84,↓,88,↓,各地,生,"11,550","5,345","2,100",99,92
7,2004,1,3,2004013,tuna_next,1,マグロ,,,,,,海外,生,"7,350","2,605","1,680",109,75
8,2004,1,3,2004013,bigeye,0,メバチ,105.1,107,,94,,各地*,冷凍,"4,200","1,049",630,99,112
9,2004,1,4,2004014,tuna,0,マグロ,35.6,119,↑,125,↑,各地,生,"26,250","5,193","1,575",97,79


In [44]:
#drop week_trend, increase_decrease_1, year_over_year_comparison, increase_decrease_2
df = df.drop(columns=["week_trend", "increase_decrease_1", "year_over_year_comparison", "increase_decrease_2"])

In [46]:
df.head(10)

,year,month,week,code,row_type,sub_row,item,volume_tonnes,origin,grade,price_high,price_mid,price_low,price_vs_prev_week_pct,price_vs_prev_year_pct
0,2004,1,1,2004011,tuna,0,マグロ,49.2,各地,生,"27,300","6,009","2,100",108,94
1,2004,1,1,2004011,tuna_next,1,マグロ,,海外,生,"9,450","2,987","1,890",112,76
2,2004,1,1,2004011,bigeye,0,メバチ,97.9,各地*,冷凍,"4,200","1,095",578,100,111
3,2004,1,2,2004012,tuna,0,マグロ,35.8,各地,生,"8,400","5,384","2,625",90,66
4,2004,1,2,2004012,tuna_next,1,マグロ,,海外,生,"5,775","2,387","1,785",80,66
5,2004,1,2,2004012,bigeye,0,メバチ,97.9,各地*,冷凍,"4,200","1,055",630,96,111
6,2004,1,3,2004013,tuna,0,マグロ,30.0,各地,生,"11,550","5,345","2,100",99,92
7,2004,1,3,2004013,tuna_next,1,マグロ,,海外,生,"7,350","2,605","1,680",109,75
8,2004,1,3,2004013,bigeye,0,メバチ,105.1,各地*,冷凍,"4,200","1,049",630,99,112
9,2004,1,4,2004014,tuna,0,マグロ,35.6,各地,生,"26,250","5,193","1,575",97,79
